In [36]:
import os
from dotenv import load_dotenv
load_dotenv("/workspaces/MY-RAG-application-Chayakorn/.env", override=True)
from google import genai

load_dotenv()

True

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # Loads environment variables from .env
api_key = os.getenv("GEMINI_API_KEY")

In [4]:
import requests
#retrive the data from the URL
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
Data = response.json()

In [5]:
Data

[{'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 48},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 460},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 397},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 153},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249}]

In [6]:

#in real cases you need to preprocess data from the URL or website
documents = []
url_prefix = "https://datatalks.club/faq"
# Loop through each course in the Data list and retrieve the course data
for course in Data:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status() #if some error occurs, Dont continue to the next step, raise an error
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1400

In [7]:
documents[5]

{'id': '833fbaa15f',
 'course': 'ai-dev-tools-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What are the prerequisites and system requirements for this course?',
 'answer': "You don't need any previous experience with AI tools. A basic ability to program (Python, JavaScript, or similar) is enough to follow the materials and complete the projects."}

In [8]:
from minsearch import Index #THere are many other libraries for indexing and searching, you can choose any library you want. I just use this library as an example.
#minesearch by Alexey Grigorev
index = Index( # depending on json structure, you can change the text_fields and keyword_fields to fit your data
    text_fields=["question", "section", "answer"], 
    keyword_fields=["course"]
)

index.fit(documents)

In [9]:
def search(question, course):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}
    
    return index.search(
        question, 
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [10]:
question = "I just discovered the course. Can I join now?"

In [11]:
search_results = search(question, "llm-zoomcamp")
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [12]:
Instruction = f"""Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."""

In [13]:
def build_context(search_results): # make the context(search result) more readable for the LLM
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [14]:
context=build_context(search_results)

In [15]:
USER_PROMPT_TEMPLATE = f"""
Question: {question} 
Answer in thai

Context: {context}
"""

In [16]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [17]:
prompt = build_prompt(question, search_results)
print(prompt)

Question: I just discovered the course. Can I join now? 
Answer in thai

Context: General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimentin

In [18]:
api_key = os.getenv("GEMINI_API_KEY")

In [19]:
genai_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [46]:
from google.genai import types

message_history = [
    types.Content(role="user", parts=[types.Part(text=prompt)])
]

response = genai_client.models.generate_content(
    model="gemini-3.5-flash",
    contents=message_history,
    config=genai.types.GenerateContentConfig(system_instruction=Instruction)
)

In [47]:
print(response.text)

ได้ครับ/ค่ะ แต่หากคุณต้องการได้รับใบประกาศนียบัตร คุณจำเป็นต้องส่งโปรเจกต์ของคุณในช่วงที่ยังคงเปิดรับการส่งผลงานอยู่


In [48]:
print(response.model_dump_json)

<bound method BaseModel.model_dump_json of GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text='ได้ครับ/ค่ะ แต่หากคุณต้องการได้รับใบประกาศนียบัตร คุณจำเป็นต้องส่งโปรเจกต์ของคุณในช่วงที่ยังคงเปิดรับการส่งผลงานอยู่',
            thought_signature=b'\x12\xb5\x12\n\xb2\x12\x01i\x14}\x13\x8b\x11\x91\x8d6\x98\xa4\xb8\xbdy\x1dp"\xe2_W\x05\xff;\xd7\xddG\xf2\x92\xb1\xd2\'.\x86\xa1\xfb\xe4\xe6D\xcd\x92\xfb\xbf\xf3W^\xd0y\xa6\xfc\xd9\x0e\x10\x7fW\xaa\x1ey\xd6\x01%2H\xb9#~\xe9e\xf6C[F\x10.\xc9\xdfW\x96O\x9a\xbe\x0f\x98\x84\x83\x80\xef\xab\xc8\x92y...'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-3.5-flash',
  response_id='u8K8as7oCLrDg8UP3MDMmAg',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidat

In [ ]:
# input_price = 0.75 / 1_000_000
# output_price = 4.50 / 1_000_000

# cost = (
#     response.usage.input_tokens * input_price +
#     response.usage.output_tokens * output_price
# )

# cost
#Calculate the cost of the API call based on input and output tokens for chatgpt

In [50]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [49]:
def llm(Instruction, prompt, model="gemini-3.5-flash"):
 message_history = [
     types.Content(role="user", parts=[types.Part(text=prompt)])
 ]

 response = genai_client.models.generate_content(
    model=model,
    contents=message_history,
    config=types.GenerateContentConfig(system_instruction=Instruction)
 )
 return response.text

In [51]:
def rag(query, model="gemini-3.5-flash"):
    search_results = search(query,"llm-zoomcamp")
    prompt = build_prompt(query, search_results)
    answer = llm(Instruction, prompt, model=model)
    return answer

In [52]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

ได้ค่ะ/ครับ คุณยังสามารถเข้าร่วมได้ แต่หากคุณต้องการได้รับใบรับรอง (certificate) คุณจำเป็นต้องส่งโปรเจกต์ของคุณในช่วงเวลาที่ยังคงเปิดรับการส่งผลงานอยู่
